In [ ]:
# Import pyarrow first to fix extension type registration issue
import pyarrow
import pyarrow.parquet

import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path
import os
import json
from dataclasses import dataclass
import matplotlib.pyplot as plt
from functools import reduce
from scipy.stats import pearsonr
import seaborn as sns
from matplotlib.lines import Line2D
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from typing import Optional

from moral_lens.models import load_model_config, update_model_config_cache

sns.set_style(style="whitegrid")

pd.set_option('display.max_columns', None)
FIGURE_SAVE_DIR = "data/figures/"


TAXONOMY_MACRO = {
    "Consequentialism": ["MaxDependents", "MaxFutureContribution", "MaxHope",
                        "MaxLifeLength", "MaxNumOfLives", "SaveTheStrong",
                        "MaxInspiration", "MaxPastContribution"],
    "Deontology":       ["SaveTheUnderprivileged", "Egalitarianism",
                         "SaveTheVulnerable", "AnimalRights", "PickRandomly",
                         "AppealToLaw", "RetributiveJustice", "FavorHumans"],
    "Other":            ["Other"],
    # "Refusal":          ["Refusal", ""],
}
macro_map = {
    fine: macro
    for macro, fines in TAXONOMY_MACRO.items()
    for fine in fines
}


ORDER = {
    "Age": ["Young", "Old"],
    "Fitness": ["Unfit", "Fit"],
    "Gender": ["Female", "Male"],
    "SocialValue": ["Low", "High"],
}
FLAT_ORDER = ['Overall'] + [element for k, vs in ORDER.items() for element in [k, *vs]]


# helper to convert p to stars
def p_to_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else: return ''


Note: In order to run the code below, you need to download their results file and put it to data/20250507/.
(Data: https://drive.google.com/file/d/1Y56QBjnw5Y-E7FjHi0isSRPdtOX4-xtt/view?usp=share_link)

In [ ]:
# Workaround for pyarrow extension type registration issue
try:
    dfs = pd.read_parquet("../data/20250507/all_model_runs.parquet")
except Exception as e:
    if "arrow.py_extension_type" in str(e):
        # Clear any existing extension type registrations and retry
        import pyarrow as pa
        try:
            pa.unregister_extension_type("arrow.py_extension_type")
        except:
            pass
        dfs = pd.read_parquet("../data/20250507/all_model_runs.parquet")
    else:
        raise

print(f"Dataframe shape: {dfs.shape}")
dfs.head(2)

In [5]:
# Let's check how many responses we actually have for individual models
response_counts = dfs.groupby('model_name').size().reset_index(name='response_count')
response_counts = response_counts.sort_values('response_count', ascending=False)

print(f"Total number of models: {len(response_counts)}")
print(f"\nResponse counts per model:")
print(response_counts.to_string(index=False))

# Summary statistics
print(f"\nSummary statistics:")
print(f"  Mean responses per model: {response_counts['response_count'].mean():.1f}")
print(f"  Median responses per model: {response_counts['response_count'].median():.1f}")
print(f"  Min responses per model: {response_counts['response_count'].min()}")
print(f"  Max responses per model: {response_counts['response_count'].max()}")
print(f"  Std responses per model: {response_counts['response_count'].std():.1f}")



Total number of models: 85

Response counts per model:
                   model_name  response_count
             Claude 3.5 Haiku            6400
                 Llama 3.1 8B            6400
             Llama 4 Maverick            6400
                Llama 4 Scout            6400
     Mistral 7B Instruct v0.1            6400
     Mistral 7B Instruct v0.3            6400
                Mistral Large            6400
           Mistral Large 2407            6400
                 Mistral Nemo            6400
                Mistral Small            6400
            Claude 3.5 Sonnet            6400
                   Nova Micro            6400
                     Nova Pro            6400
                        Phi-4            6400
                 Qwen 1.5 14B            6400
                 Qwen 1.5 32B            6400
                  Qwen 1.5 4B            6400
                 Qwen 1.5 72B            6400
                  Qwen 1.5 7B            6400
                   Qwen 2

In [6]:
# Check dataset balance for a selected model: 
# 1. Do all conditions appear equally often in option A (choice1) vs B (choice2)?
# 2. Are there correlations between different attributes?

from scipy.stats import chi2_contingency
from itertools import combinations

# Select a model to analyze
# You can change this to any model name from the response_counts above
selected_model = response_counts.iloc[0]['model_name']  # Default to first model
# Or uncomment and set manually:
# selected_model = "gpt-4o-mini-2024-07-18"

print(f"Analyzing balance for model: {selected_model}")
print("=" * 80)

# Filter data for selected model
df_model = dfs[dfs['model_name'] == selected_model].copy()
print(f"Total responses for this model: {len(df_model)}")

# Get unique scenarios for this model (each two_choices represents a unique scenario)
unique_scenarios = df_model[['two_choices', 'phenomenon_category', 'category1', 'category2']].drop_duplicates()

print("=" * 80)
print("1. BALANCE CHECK: Attribute values in Choice1 vs Choice2")
print("=" * 80)

# For each phenomenon category, check if values appear equally in category1 vs category2
for phenom in ORDER.keys():
    if phenom not in unique_scenarios['phenomenon_category'].values:
        continue
    
    phenom_data = unique_scenarios[unique_scenarios['phenomenon_category'] == phenom]
    
    print(f"\n{phenom}:")
    print("-" * 60)
    
    # Get all possible values for this phenomenon
    all_values = set(phenom_data['category1'].unique()) | set(phenom_data['category2'].unique())
    
    for value in sorted(all_values):
        # Count how many times this value appears in category1 vs category2
        in_choice1 = (phenom_data['category1'] == value).sum()
        in_choice2 = (phenom_data['category2'] == value).sum()
        total = in_choice1 + in_choice2
        
        if total > 0:
            pct_choice1 = (in_choice1 / total) * 100
            pct_choice2 = (in_choice2 / total) * 100
            imbalance = abs(pct_choice1 - pct_choice2)
            
            status = "✓ BALANCED" if imbalance < 5 else "⚠ IMBALANCED"
            print(f"  {value:15s}: Choice1={in_choice1:4d} ({pct_choice1:5.1f}%) | "
                  f"Choice2={in_choice2:4d} ({pct_choice2:5.1f}%) | {status}")

print("\n" + "=" * 80)
print("2. CORRELATION CHECK: Between different attributes")
print("=" * 80)

# Check for correlations between different attributes
# We'll create a matrix showing if different attributes co-occur more than expected

# Get all unique scenarios with their attributes
scenario_attrs = unique_scenarios[['two_choices', 'phenomenon_category', 'category1', 'category2']].copy()

# Create a pivot table to check correlations
# For each scenario, we want to know which attributes are present
attr_presence = {}

for _, row in scenario_attrs.iterrows():
    scenario_id = row['two_choices']
    phenom = row['phenomenon_category']
    val1 = row['category1']
    val2 = row['category2']
    
    if scenario_id not in attr_presence:
        attr_presence[scenario_id] = {}
    
    # Store both values for this attribute
    attr_presence[scenario_id][f"{phenom}_choice1"] = val1
    attr_presence[scenario_id][f"{phenom}_choice2"] = val2

# Convert to DataFrame
attr_df = pd.DataFrame.from_dict(attr_presence, orient='index')

# Check correlations between different phenomenon categories
phenom_categories = unique_scenarios['phenomenon_category'].unique()

print("\nChecking for correlations between different attributes...")
print("(Looking at whether scenarios with one attribute are more likely to have another)\n")

# For each pair of attributes, check if they're independent
correlation_results = []

for phenom1, phenom2 in combinations(phenom_categories, 2):
    if phenom1 not in ORDER or phenom2 not in ORDER:
        continue
    
    # Create contingency table: does scenario have phenom1 values vs phenom2 values?
    # We'll check if the presence of specific values in one attribute correlates with the other
    
    # Get scenarios that have both attributes
    scenarios_with_both = scenario_attrs[
        (scenario_attrs['phenomenon_category'] == phenom1) | 
        (scenario_attrs['phenomenon_category'] == phenom2)
    ]['two_choices'].unique()
    
    # For simplicity, let's check if scenarios are more likely to have certain combinations
    # We'll look at the distribution of values across scenarios
    
    phenom1_scenarios = set(scenario_attrs[scenario_attrs['phenomenon_category'] == phenom1]['two_choices'])
    phenom2_scenarios = set(scenario_attrs[scenario_attrs['phenomenon_category'] == phenom2]['two_choices'])
    
    # Check overlap
    overlap = len(phenom1_scenarios & phenom2_scenarios)
    total_phenom1 = len(phenom1_scenarios)
    total_phenom2 = len(phenom2_scenarios)
    total_unique = len(phenom1_scenarios | phenom2_scenarios)
    
    # Expected overlap if independent
    expected_overlap = (total_phenom1 * total_phenom2) / total_unique if total_unique > 0 else 0
    
    if total_unique > 0:
        correlation_ratio = overlap / expected_overlap if expected_overlap > 0 else 0
        correlation_results.append({
            'Attribute1': phenom1,
            'Attribute2': phenom2,
            'Overlap': overlap,
            'Expected': f"{expected_overlap:.1f}",
            'Ratio': f"{correlation_ratio:.2f}",
            'Status': "Independent" if abs(correlation_ratio - 1.0) < 0.1 else "Correlated"
        })

if correlation_results:
    corr_df = pd.DataFrame(correlation_results)
    print(corr_df.to_string(index=False))
else:
    print("No correlation analysis available (insufficient data)")

print("\n" + "=" * 80)
print("3. DETAILED BALANCE: Per unique scenario")
print("=" * 80)

# Count how many times each unique scenario appears for this model
scenario_counts = df_model.groupby('two_choices').size().reset_index(name='count')
print(f"\nTotal unique scenarios for this model: {len(scenario_counts)}")
print(f"Scenarios appear {scenario_counts['count'].min()} to {scenario_counts['count'].max()} times")
print(f"Mean appearances per scenario: {scenario_counts['count'].mean():.1f}")
print(f"Std appearances per scenario: {scenario_counts['count'].std():.1f}")

# Check if scenarios are balanced across phenomenon categories
phenom_dist = unique_scenarios['phenomenon_category'].value_counts().sort_index()
print(f"\nDistribution of scenarios by phenomenon category:")
for phenom, count in phenom_dist.items():
    print(f"  {phenom:15s}: {count:4d} scenarios")



Analyzing balance for model: Claude 3.5 Haiku
Total responses for this model: 6400
1. BALANCE CHECK: Attribute values in Choice1 vs Choice2

Age:
------------------------------------------------------------
  Old            : Choice1=  48 ( 50.0%) | Choice2=  48 ( 50.0%) | ✓ BALANCED
  Young          : Choice1=  48 ( 50.0%) | Choice2=  48 ( 50.0%) | ✓ BALANCED

Fitness:
------------------------------------------------------------
  Fit            : Choice1=  48 ( 50.0%) | Choice2=  48 ( 50.0%) | ✓ BALANCED
  Unfit          : Choice1=  48 ( 50.0%) | Choice2=  48 ( 50.0%) | ✓ BALANCED

Gender:
------------------------------------------------------------
  Female         : Choice1=  56 ( 50.0%) | Choice2=  56 ( 50.0%) | ✓ BALANCED
  Male           : Choice1=  56 ( 50.0%) | Choice2=  56 ( 50.0%) | ✓ BALANCED

SocialValue:
------------------------------------------------------------
  High           : Choice1= 168 ( 50.0%) | Choice2= 168 ( 50.0%) | ✓ BALANCED
  Low            : Choice1= 168